In [6]:
from dotenv import load_dotenv
import os
from ollama import Client

load_dotenv()

client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.environ.get('OLLAMA_API_KEY')}
)

In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [9]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

**Yes—you can still join the course.** If you want to earn a certificate, just make sure you submit your capstone project before the submission window closes. Otherwise, you’re welcome to start learning right away!

'**Yes—you can still join the course.** If you want to earn a certificate, just make sure you submit your capstone project before the submission window closes. Otherwise, you’re welcome to start learning right away!'

In [10]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model we will use for semantic similarity.
model = SentenceTransformer("all-MiniLM-L6-v2")


from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)